In [ ]:
import numpy as np
import emcee
import matplotlib.pyplot as plt

This is an implementation of the Voigt-Reuss-Hill model on the original probablistic inversion with mcmc.

In [ ]:
# ----- Fluid End-Member Properties -----
K_water = 2.2e9      # Bulk modulus of water, Pa
rho_water = 1000.0   # Density of water, kg/m³

K_gas = 0            # Bulk modulus for gas, Pa (Voigt mixing)
rho_gas = 0.020      # Density for gas, kg/m³

In [ ]:
# Voigt upper bound (isostrain average)
# RPH Section 4.2, p.174: M_V = sum(f_i * M_i)
def voigt_bulk(K_s, K_f, phi):
    return phi * K_f + (1 - phi) * K_s

In [ ]:
# Reuss lower bound (isostress average)
# RPH Section 4.2, p.174: 1/M_R = sum(f_i / M_i)
def reuss_bulk(K_s, K_f, phi):
    return 1.0 / (phi / K_f + (1 - phi) / K_s)

In [ ]:
# Voigt-Reuss-Hill average
# RPH Section 4.4, p.177: M_VRH = (M_V + M_R) / 2
def hill_bulk(K_s, K_f, phi):
    K_V = voigt_bulk(K_s, K_f, phi)
    K_R = reuss_bulk(K_s, K_f, phi)
    return 0.5 * (K_V + K_R)

In [ ]:
# Effective shear modulus using Hill average
# RPH Section 4.4, p.177: For fluid (G_f = 0), Voigt = (1-phi)*G_s,
# Reuss = 0, so Hill = 0.5*(1-phi)*G_s
def effective_shear_modulus(G_s, phi):
    return 0.5 * (1 - phi) * G_s

In [ ]:
def effective_density(rho_s, rho_f, phi):
    return (1 - phi) * rho_s + phi * rho_f

In [ ]:
# Seismic velocities from effective moduli
# RPH Section 3.1, p.81: Vp = sqrt((K + 4/3*G) / rho), Vs = sqrt(G / rho)
def seismic_velocities(K_eff, G_eff, rho_eff):
    Vp = np.sqrt((K_eff + (4.0/3.0) * G_eff) / rho_eff)
    Vs = np.sqrt(G_eff / rho_eff)
    return Vp, Vs

In [ ]:
def forward_model(x):
    """
    Forward model mapping unknown parameters to effective seismic properties.
    
    x = [S_w, phi, K_s, G_s, rho_s]
      S_w: water saturation (0 to 1)
      phi: crack porosity (volume fraction)
      K_s: matrix bulk modulus (Pa)
      G_s: matrix shear modulus (Pa)
      rho_s: matrix density (kg/m³)
      
    Fluid properties are computed as:
      K_f = S_w*K_water + (1-S_w)*K_gas
      rho_f = S_w*rho_water + (1-S_w)*rho_gas
      
    Then:
      K_eff = hill_bulk(K_s, K_f, phi)
      G_eff = effective_shear_modulus(G_s, phi)
      rho_eff = effective_density(rho_s, rho_f, phi)
      Vp, Vs = seismic_velocities(K_eff, G_eff, rho_eff)
      
    Returns: [Vp, Vs, rho_eff]
    """
    S_w, phi, K_s, G_s, rho_s = x
    K_f = S_w * K_water + (1 - S_w) * K_gas
    rho_f = S_w * rho_water + (1 - S_w) * rho_gas
    K_eff = hill_bulk(K_s, K_f, phi)
    G_eff = effective_shear_modulus(G_s, phi)
    rho_eff = effective_density(rho_s, rho_f, phi)
    Vp, Vs = seismic_velocities(K_eff, G_eff, rho_eff)
    return np.array([Vp, Vs, rho_eff])

In [ ]:
Vp_obs = 4700.0       # m/s (5 km/s)
Vs_obs = 2700.0       # m/s (2.8 km/s)
rho_obs = 2589
sigma_Vp = 300.0       # m/s
sigma_Vs = 100.0       # m/s
sigma_rho = 157.0      # kg/m³

In [ ]:
def log_likelihood(x):
    model = forward_model(x)  # [Vp, Vs, rho_eff]
    diff = model - np.array([Vp_obs, Vs_obs, rho_obs])
    chi2 = (diff[0]**2/sigma_Vp**2 +
            diff[1]**2/sigma_Vs**2 +
            diff[2]**2/sigma_rho**2)
    return -0.5 * chi2

In [ ]:
def log_prior(x):
    S_w, phi, K_s, G_s, rho_s = x
    if (0.0 < S_w < 1.0 and 
        0.001 < phi < 0.5 and 
        756e8 < K_s < 80e9 and
        256e8 < G_s < 40e9 and
        2680 < rho_s < 2900):
        return 0.0
    return -np.inf

In [ ]:
def log_probability(x):
    x = np.asarray(x).ravel()
    lp = log_prior(x)
    if not np.isfinite(lp):
        return -np.inf, np.full(3, np.nan)
    ll = log_likelihood(x)
    blob = forward_model(x)  # Blob: [Vp, Vs, rho_eff]
    return lp + ll, blob

In [ ]:
ndim = 5  # Unknown parameters: [S_w, phi, K_s, G_s, rho_s]
nwalkers = 3 * ndim
initial = np.zeros((nwalkers, ndim))
initial[:, 0] = np.random.uniform(0.0, 1.0, nwalkers)         # S_w
initial[:, 1] = np.random.uniform(0.001, 0.5, nwalkers)        # φ
initial[:, 2] = np.random.uniform(756e8, 80e9, nwalkers)         # K_s
initial[:, 3] = np.random.uniform(256e8, 40e9, nwalkers)         # G_s
initial[:, 4] = np.random.uniform(2680, 2900, nwalkers)         # ρ_s

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
nsteps = 50000
sampler.run_mcmc(initial, nsteps, progress=True)

# Retrieve flattened chain and blobs.
samples = sampler.get_chain(flat=True)
blobs = np.array(sampler.get_blobs(flat=True))  # Each blob: [Vp, Vs, rho_eff]

# Identify best-fit sample.
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]
best_model = forward_model(best_params)

# The blobs array contains the effective seismic properties at each MCMC step.
# You can use them to construct PDFs of Vp, Vs, and effective density.
blobs = np.array(blobs)  # Ensure homogeneous shape; expected shape: (nsteps*nwalkers, 3)
Vp_pdf = blobs[:, 0]
Vs_pdf = blobs[:, 1]
rho_pdf = blobs[:, 2]

labels = [
    "Water Saturation",         #S_w
    "Crack Porosity",           # φ
    "Matrix Bulk Modulus (Pa)", # K_m
    "Matrix Shear Modulus (Pa)",# μ_m
    "Matrix Density (kg/m³)"    # ρ_m
]

fig, axes = plt.subplots(ndim, figsize=(10, 14), sharex=True)
for i in range(ndim):
    axes[i].plot(samples[:, i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()

In [ ]:
def plot_1d_histograms(samples, labels=None, bins=100):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]  
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    for i in range(n_parameters):
        ax = axes[i] if n_parameters > 1 else axes  
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value") if n_parameters > 1 else axes.set_xlabel("Parameter Value")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_1d_histograms(samples, labels = labels, bins=50)

In [ ]:
param_names = list(labels)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('Crack Porosity')
idx_water    = param_names.index('Water Saturation')

# Function to plot and save a single histogram
def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()

# --- Sanity check before saving ---
def summarize_array(name, arr):
    arr = np.asarray(arr)
    print(f"\n{name}: type={type(arr)}, dtype={arr.dtype}, shape={arr.shape}")
    if arr.size == 1:
        print(f"⚠️  Only one element! value={arr}")
    else:
        print(f"✅  Multi-sample array OK — n={arr.size}, min={np.min(arr):.4g}, max={np.max(arr):.4g}")
    return arr
print(idx_porosity)
idx_porosity = summarize_array("idx_porosity", idx_porosity)
idx_water    = summarize_array("idx_water", idx_water)
np.save("saturation_vrh.npy", np.asarray(idx_water))
np.save("porosity_vrh.npy", np.asarray(idx_porosity))
# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='porosity_vrh_5.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='saturation_vrh_5.png')

In [ ]:
bb1 = blobs.reshape(750000, 3)
post_lab = ['vp', 'vs', 'rho_e']
plot_1d_histograms(bb1, labels=post_lab, bins=100)

In [ ]:

def plot_2d_color_plots_vrh(samples, labels):
    """
    Plot 2D color maps (histograms) for each unique pair of MCMC parameters 
    from a VRH inversion (ndim=6).

    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      labels:  list of parameter labels (length ndim).
    """
    ndim = samples.shape[1]
    
    # Extract each parameter's chain data into a list.
    variable_data = [samples[:, i] for i in range(ndim)]
    
    # Create all unique pairs (i, j) with i < j.
    pairs = [(i, j) for i in range(ndim) for j in range(i+1, ndim)]
    npairs = len(pairs)
    
    # Determine grid size (roughly square).
    ncols = int(np.ceil(np.sqrt(npairs)))
    nrows = int(np.ceil(npairs / ncols))
    
    fig, axs = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axs = np.ravel(axs)  # Flatten axes array.
    
    for idx, (i, j) in enumerate(pairs):
        # Use the actual data arrays for each parameter.
        x_data = variable_data[i]
        y_data = variable_data[j]
        
        # Compute 2D histogram (100 bins per axis).
        hist, x_edges, y_edges = np.histogram2d(x_data, y_data, bins=100)
        
        # Plot the histogram as a color map.
        im = axs[idx].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                              origin="lower", aspect="auto", cmap="viridis")
        axs[idx].set_xlabel(labels[i])
        axs[idx].set_ylabel(labels[j])
        axs[idx].set_title(f"{labels[j]} vs {labels[i]}")
    
    # Turn off any unused subplots.
    for k in range(idx+1, len(axs)):
        axs[k].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs.tolist(), orientation="horizontal", fraction=0.02, pad=0.04)
    plt.show()


# Assuming 'samples_vrh' is your flattened MCMC chain from the VRH inversion (shape: (N, 6)):
plot_2d_color_plots_vrh(samples, labels)

In [ ]:
def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[0], best_params[1]))

In [ ]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

phi_mode = marginal_mode(samples[:,1], bins=100)
S_w_mode = marginal_mode(samples[:,0], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))

In [ ]:
# --- Distribution center (mean/median) based water estimate ---
phi_mean   = np.mean(samples[:,1])
S_w_mean   = np.mean(samples[:,0])
phi_median = np.median(samples[:,1])
S_w_median = np.median(samples[:,0])

print("Mean phi:", phi_mean)
print("Mean S_w:", S_w_mean)
print("Mean-based water estimate (m):", W_thickness(S_w_mean, phi_mean))

print("\nMedian phi:", phi_median)
print("Median S_w:", S_w_median)
print("Median-based water estimate (m):", W_thickness(S_w_median, phi_median))

In [ ]:
# --- Distribution-based water layer thickness ---
# Compute thickness for EVERY MCMC sample, then take statistics
thickness_samples = 8500 * samples[:,0] * samples[:,1]  # 8500 * S_w * phi

print('=== Distribution-based water layer thickness ===')
print('Mean thickness (m):', np.mean(thickness_samples))
print('Median thickness (m):', np.median(thickness_samples))
print('Mode thickness (m):', marginal_mode(thickness_samples, bins=100))
print('Std thickness (m):', np.std(thickness_samples))
print('95% CI (m):', np.percentile(thickness_samples, [2.5, 97.5]))
print('5th percentile (m):', np.percentile(thickness_samples, 5))
print('95th percentile (m):', np.percentile(thickness_samples, 95))

# Save samples array to disk for post-processing
np.save(os.path.join(output_dir, f'samples_{case_name}.npy'), samples)
np.save(os.path.join(output_dir, f'thickness_samples_{case_name}.npy'), thickness_samples)
print(f'Saved samples and thickness arrays to {output_dir}')